In [0]:
    # Databricks notebook source
# Landing (Parquet) -> Bronze (Delta)

# -------------------------------
# Widgets for parameterization
# -------------------------------
dbutils.widgets.text("domain",           "CM")
dbutils.widgets.text("storageAccName",   "stclinical")
dbutils.widgets.text("srcContainerName", "landing")
dbutils.widgets.text("tgtCatalog",       "clintrail_dev")
dbutils.widgets.text("tgtSchema",        "clintrail_bronze")
dbutils.widgets.text("tgtTableName",     "cm")
dbutils.widgets.text("secretScope",      "clinical-scope")
dbutils.widgets.text("secretKey",        "stclinical-storage-key")




In [0]:
domain    = dbutils.widgets.get("domain")
account   = dbutils.widgets.get("storageAccName")
container = dbutils.widgets.get("srcContainerName")

# Set storage account access key for ABFS
spark.conf.set(
    f"fs.azure.account.key.{account}.dfs.core.windows.net",
    dbutils.secrets.get(scope=dbutils.widgets.get("secretScope"), key=dbutils.widgets.get("secretKey"))
)

DATE_COL = "bronze_ingest_date"
ROOT     = f"abfss://{container}@{account}.dfs.core.windows.net/src/{domain}"
TARGET   = f"{dbutils.widgets.get('tgtCatalog')}.{dbutils.widgets.get('tgtSchema')}.{dbutils.widgets.get('tgtTableName')}"

print("ROOT:", ROOT)
print("TARGET:", TARGET)

# -------------------------------
# 

In [0]:
# Identify dates to load
# -------------------------------
landing = {
    f.name.split("=", 1)[1].rstrip("/")
    for f in dbutils.fs.ls(ROOT)
    if f.name.startswith("ingest_date=")
}

loaded = {
    str(r[0])
    for r in spark.table(TARGET).select(DATE_COL).distinct().collect()
    if r[0] is not None
}

todo = sorted(landing.difference(loaded))

print(f"{domain}: landing={len(landing)} bronze={len(loaded)} to load={todo}")

if not todo:
    dbutils.notebook.exit(f"OK|{domain}|0 rows")

# -------------------------------
# 

In [0]:
# Build paths for missing dates
# -------------------------------
paths = [f"{ROOT}/ingest_date={d}" for d in todo]
print("Paths to load:", paths)

# -------------------------------
# Read Parquet into DataFrame
# -------------------------------
df = (
    spark.read
         .option("basePath", ROOT)
         .parquet(*paths)   # * expands list into arguments
         .withColumnRenamed("ingest_date", DATE_COL)
)

n = df.count()
print(f"Read {n} rows")

# -------------------------------
# Prepare replaceWhere predicate
# -------------------------------
date_list = ", ".join(f"'{d}'" for d in todo)
predicate = f"{DATE_COL} in ({date_list})"
print("replaceWhere:", predicate)

# -------------------------------
# Write to Bronze Delta table
# -------------------------------
(df.write.format("delta")
   .mode("overwrite")
   .option("mergeSchema", "true")
   .option("replaceWhere", predicate)
   .saveAsTable(TARGET))

print(f"Wrote {n} rows to {TARGET}")

In [0]:
spark.conf.get("fs.azure.account.key.stclinical.dfs.core.windows.net")

In [0]:
# NOTE: Hardcoded storage account key removed — now read from Databricks Secrets in Cell 2.
# Setup steps:
#   1. Create a Databricks Secret Scope:  databricks secrets create-scope clinical-scope
#   2. Add the storage key:               databricks secrets put-secret clinical-scope stclinical-storage-key
#   3. Set widgets "secretScope" and "secretKey" when running the notebook.
# IMPORTANT: Rotate the Azure Storage Account Access Key for stclinical immediately —
#            the previous key was committed to Git and must be considered compromised.
